In [30]:
from bs4 import BeautifulSoup as BS
import pandas as pd
import requests 

In [ ]:
url= 'https://scikit-learn.org/stable/supervised_learning.html' 
page = requests.get(url)
print(page.status_code)

200


In [3]:
soup = BS(page.content, "html.parser")

In [4]:
categores = soup.find_all("li", class_="toctree-l1")

In [5]:
links = []

for li in categores:
    a = li.find("a", href=True)
    if a:
        links.append(a["href"])

print(links) 

['#', 'unsupervised_learning.html', 'model_selection.html', 'metadata_routing.html', 'inspection.html', 'visualizations.html', 'callbacks.html', 'data_transforms.html', 'datasets.html', 'computing.html', 'model_persistence.html', 'common_pitfalls.html', 'data_interoperability.html', 'machine_learning_map.html', 'presentations.html', 'modules/linear_model.html', 'modules/lda_qda.html', 'modules/kernel_ridge.html', 'modules/svm.html', 'modules/sgd.html', 'modules/neighbors.html', 'modules/gaussian_process.html', 'modules/cross_decomposition.html', 'modules/naive_bayes.html', 'modules/tree.html', 'modules/ensemble.html', 'modules/multiclass.html', 'modules/feature_selection.html', 'modules/semi_supervised.html', 'modules/isotonic.html', 'modules/calibration.html', 'modules/neural_networks_supervised.html']


In [ ]:
from urllib.parse import urljoin

BASE_URL = "https://scikit-learn.org/stable/"
START_URL = urljoin(BASE_URL, "user_guide.html")

visited = set()
seen_text = set()
rows = []

tags = ["h1", "h2", "h3", "p", "pre"]

def crawl(url, depth):

    if depth > 4:
        return

    if url in visited:
        return

    visited.add(url)

    try:
        page = requests.get(url, timeout=10)
        if page.status_code != 200:
            return
    except:
        return

    soup = BS(page.text, "html.parser")

    for tag in tags:
        for element in soup.find_all(tag):
            text = element.get_text(" ", strip=True)

            if text and text not in seen_text:
                seen_text.add(text)
                rows.append({
                    "tag": tag,
                    "text": text
                })

    for a in soup.find_all("a", class_="reference internal", href=True):
        next_url = urljoin(BASE_URL, a["href"])

        if next_url.startswith(BASE_URL):
            crawl(next_url, depth + 1)

crawl(START_URL, 0)

df = pd.DataFrame(rows)

print(f"Pages visited: {len(visited)}")
print(df.shape)

Depth 0: https://scikit-learn.org/stable/user_guide.html
Depth 1: https://scikit-learn.org/stable/supervised_learning.html
Depth 2: https://scikit-learn.org/stable/modules/linear_model.html
Depth 3: https://scikit-learn.org/stable/lda_qda.html
Depth 3: https://scikit-learn.org/stable/kernel_ridge.html
Depth 3: https://scikit-learn.org/stable/svm.html
Depth 3: https://scikit-learn.org/stable/sgd.html
Depth 3: https://scikit-learn.org/stable/neighbors.html
Depth 3: https://scikit-learn.org/stable/gaussian_process.html
Depth 3: https://scikit-learn.org/stable/cross_decomposition.html
Depth 3: https://scikit-learn.org/stable/naive_bayes.html
Depth 3: https://scikit-learn.org/stable/tree.html
Depth 3: https://scikit-learn.org/stable/ensemble.html
Depth 3: https://scikit-learn.org/stable/multiclass.html
Depth 3: https://scikit-learn.org/stable/feature_selection.html
Depth 3: https://scikit-learn.org/stable/semi_supervised.html
Depth 3: https://scikit-learn.org/stable/isotonic.html
Depth 3: h

In [37]:
df

,tag,text
0,h1,User Guide #
1,p,Section Navigation
2,p,previous
3,p,Installing scikit-learn
4,p,next
...,...,...
12433,pre,>>> from sklearn.datasets import load_iris >>>...
12434,pre,">>> from sklearn.metrics import make_scorer , ..."
12435,h1,sklearn.ensemble #
12436,p,"Ensemble-based methods for classification, reg..."


In [44]:
df.to_csv("sklearn_data.csv", index=False, encoding="utf-8")